# 01 · NumPy: computação vetorizada

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Pandas, scikit-learn e PyTorch são construídos sobre a mesma ideia: guardar números em blocos contíguos de memória e operar sobre o bloco inteiro de uma vez. O NumPy é onde essa ideia aparece de forma mais direta. Dominar o *array* é o que permite ler mensagens de erro de formato, escrever código rápido e, mais adiante, entender tensores.

## O que você será capaz de fazer ao final

1. Criar *arrays* e descrever o seu formato, o seu tipo e o quanto ocupam.
2. Selecionar dados com fatias, máscaras e listas de índices, sabendo quando o resultado é uma visão e quando é uma cópia.
3. Agregar ao longo de linhas ou de colunas com `axis`.
4. Prever o resultado do *broadcasting* e usá-lo no lugar de laços.
5. Fazer sorteios reprodutíveis e dividir dados em treino e teste.
6. Implementar um classificador k-vizinhos mais próximos sem nenhum laço.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import numpy as np

print("NumPy", np.__version__)
np.set_printoptions(precision=3, suppress=True)      # mostra 3 casas decimais e evita notação científica

## 1. Por que não usar listas?

Vamos somar os quadrados de um milhão de números de duas formas.

In [ ]:
import timeit

n = 1_000_000
lista = list(range(n))
vetor = np.arange(n)

t_lista = min(timeit.repeat(lambda: sum(x * x for x in lista), number=1, repeat=3))
t_numpy = min(timeit.repeat(lambda: (vetor * vetor).sum(), number=1, repeat=3))

print(f"Lista com laço: {t_lista * 1000:7.1f} ms")
print(f"NumPy:          {t_numpy * 1000:7.1f} ms   ({t_lista / t_numpy:.0f} vezes mais rápido)")

A diferença tem duas causas:

- uma **lista** guarda referências para objetos espalhados pela memória, cada um com o seu tipo. A cada operação o Python precisa descobrir o tipo e buscar o objeto;
- um ***array*** guarda os números lado a lado, todos do **mesmo tipo**. A operação é executada por código compilado, sobre o bloco inteiro.

Escrever `vetor * vetor`, sem laço, chama-se **vetorizar**. O ganho de velocidade é grande, mas o ganho maior é de clareza: o código fica parecido com a fórmula.

In [ ]:
import sys

print(f"Lista:  {(sys.getsizeof(lista) + n * sys.getsizeof(10**6)) / 1e6:6.1f} MB  (a lista de referências mais um objeto por número)")
print(f"Array:  {vetor.nbytes / 1e6:6.1f} MB  · tipo {vetor.dtype}")

## 2. Anatomia de um *array*

Todo *array* tem três propriedades que você vai consultar o tempo todo.

In [ ]:
A = np.array([[1.5, 2.0, 3.5],
              [4.0, 5.5, 6.0]])

print("formato (shape):", A.shape, "→ 2 linhas e 3 colunas")
print("dimensões (ndim):", A.ndim)
print("tipo (dtype):", A.dtype)
print("elementos (size):", A.size)

| Formato | Nome usual | Exemplo em ciência de dados |
|---|---|---|
| `(n,)` | vetor | uma coluna: as idades de n clientes |
| `(n, d)` | matriz | uma tabela: n linhas, d colunas |
| `(n, a, l)` | tensor de ordem 3 | n imagens em tons de cinza, de a por l pixels |

> ⚠️ `(n,)` e `(n, 1)` **não** são a mesma coisa. O primeiro é um vetor com n elementos; o segundo é uma matriz com n linhas e uma coluna. A diferença parece detalhe e é a origem de muitos erros (seção 5).

### Formas de criar

In [ ]:
print(np.zeros((2, 3)))                     # preenchido com zeros; o formato vai em uma tupla
print(np.arange(0, 10, 2))                  # como range: início, fim (fora), passo
print(np.linspace(0, 1, 5))                 # 5 pontos igualmente espaçados, com as pontas incluídas
print(np.full((2, 2), 7))                   # preenchido com um valor
print(np.eye(3, dtype=int))                 # matriz identidade

### O tipo importa

Um *array* tem um único tipo. Misturar inteiros e decimais converte tudo para decimal; o `dtype` também define a precisão e a memória.

In [ ]:
inteiros = np.array([1, 2, 3])
misto = np.array([1, 2.5, 3])
pequeno = np.array([1, 2.5, 3], dtype=np.float32)

for nome, arr in [("inteiros", inteiros), ("misto", misto), ("float32", pequeno)]:
    print(f"{nome:<9} dtype={str(arr.dtype):<8} bytes por elemento={arr.itemsize}")

print(inteiros.astype(float) / 2)           # astype converte o tipo e devolve um array novo

O padrão do NumPy é `float64`. Bibliotecas de *deep learning* usam `float32`, que ocupa metade da memória e basta para treinar redes.

### Mudar o formato

`reshape` reorganiza os mesmos elementos em outro formato. O valor `-1` pede ao NumPy que calcule aquela dimensão.

In [ ]:
v = np.arange(12)
print(v.reshape(3, 4))
print(v.reshape(-1, 2).shape, "→ o -1 virou 6")
print(v.reshape(-1, 1).shape, "→ vetor transformado em matriz de uma coluna")

### ✏️ Exercício 1 (★): criar e inspecionar

Crie:

1. `grade`: os inteiros de 0 a 23 em uma matriz de 4 linhas e 6 colunas;
2. `metades`: 11 pontos igualmente espaçados de 0 a 5 (0, 0.5, ..., 5);
3. `grade_f32`: a `grade` convertida para `float32`.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
grade = ...
metades = ...
grade_f32 = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert grade.shape == (4, 6) and grade[3, 5] == 23 and grade[1, 0] == 6, "confira o formato e a ordem dos elementos"
assert metades.shape == (11,) and np.isclose(metades[1], 0.5) and metades[-1] == 5
assert grade_f32.dtype == np.float32 and grade_f32.nbytes == grade.size * 4
print("Exercício 1 correto.")

## 3. Indexação

### Fatias

Em duas dimensões, a seleção é `A[linhas, colunas]`. Uma fatia `início:fim:passo` funciona como nas listas; `:` sozinho significa "tudo".

In [ ]:
M = np.arange(20).reshape(4, 5)
print(M, "\n")
print("linha 1:          ", M[1])
print("coluna 2:         ", M[:, 2])
print("elemento [2, 3]:  ", M[2, 3])
print("últimas 2 linhas, colunas pares:\n", M[-2:, ::2])

### Máscaras booleanas

Uma comparação devolve um *array* de `True`/`False` com o mesmo formato. Usada como índice, ela seleciona os elementos verdadeiros. É a ferramenta mais usada em análise de dados, e o pandas a herda sem mudanças.

In [ ]:
idades = np.array([23, 41, 17, 35, 68, 29])

maiores = idades >= 18
print(maiores)
print("selecionados:", idades[maiores])
print("quantos:", maiores.sum(), "· proporção:", maiores.mean())      # True vale 1
print("entre 25 e 45:", idades[(idades >= 25) & (idades <= 45)])      # & é "e", | é "ou", ~ é "não"

> ⚠️ Com *arrays*, combine condições com `&`, `|` e `~`, **com parênteses** em volta de cada comparação. As palavras `and`, `or` e `not` não funcionam elemento a elemento e geram erro.

`np.where(condição, valor_se_verdadeiro, valor_se_falso)` escolhe elemento a elemento:

In [ ]:
faixa = np.where(idades < 30, "jovem", "adulto")
print(faixa)

### Listas de índices

Passar uma lista (ou um *array*) de posições seleciona exatamente aquelas, na ordem dada e com repetição, se houver.

In [ ]:
print(idades[[0, 0, -1]])                    # posições 0, 0 e a última
ordem = np.argsort(idades)                   # posições que ordenariam o array
print("ordem crescente:", idades[ordem], "· os 2 maiores:", idades[ordem[-2:]])

### Visão ou cópia?

Este é o ponto em que o NumPy mais surpreende.

- Uma **fatia** devolve uma **visão**: outro jeito de olhar para a **mesma memória**. Alterar a visão altera o original.
- Uma **máscara** ou uma **lista de índices** devolve uma **cópia**.

In [ ]:
original = np.arange(6)

fatia = original[:3]            # visão
fatia[0] = 100
print("após alterar a fatia:  ", original)

selecao = original[[3, 4]]      # cópia
selecao[0] = -1
print("após alterar a seleção:", original)

print("fatia compartilha memória?  ", np.shares_memory(original, fatia))
print("seleção compartilha memória?", np.shares_memory(original, selecao))

Visões existem para evitar a cópia de gigabytes a cada fatiamento. Quando precisar de independência, peça explicitamente: `original[:3].copy()`.

**Atribuir** por máscara ou por lista de índices altera o original, porque aí não há objeto intermediário:

In [ ]:
valores = np.array([5.0, -2.0, 7.0, -9.0])
valores[valores < 0] = 0        # zera os negativos, no próprio array
print(valores)

### ✏️ Exercício 2 (★): limpar idades

O *array* `idades_brutas` tem valores impossíveis. Crie:

1. `validas`: máscara booleana que é verdadeira para idades entre 14 e 100, inclusive;
2. `idades_limpas`: uma **cópia** em que as idades inválidas foram trocadas por `np.nan` (o original não pode mudar);
3. `media_valida`: a média das idades válidas.

Dica: `np.nanmean` calcula a média ignorando os `nan`.

In [ ]:
# ✏️ Exercício 2
idades_brutas = np.array([34.0, 1959.0, 22.0, 0.0, 61.0, 45.0, -3.0, 28.0])
# SEU CÓDIGO AQUI
validas = ...
idades_limpas = ...
media_valida = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert validas.dtype == bool and validas.sum() == 5, "cinco idades são válidas"
assert idades_brutas[1] == 1959.0, "o array original foi alterado: trabalhe em uma cópia"
assert np.isnan(idades_limpas).sum() == 3 and idades_limpas[0] == 34.0
assert np.isclose(media_valida, 38.0)
print(f"Exercício 2 correto: média das idades válidas = {media_valida:.1f}")

## 4. Operações e agregações

Operações aritméticas e funções matemáticas atuam **elemento a elemento**.

In [ ]:
precos = np.array([100.0, 250.0, 80.0])
quantidades = np.array([2, 1, 5])

print("receita por item:", precos * quantidades)
print("com 10% de desconto:", precos * 0.9)
print("logaritmo:", np.log(precos))
print("receita total:", (precos * quantidades).sum())

### O argumento `axis`

Em uma matriz, uma agregação pode resumir tudo, cada coluna ou cada linha. O `axis` indica **a dimensão que desaparece**.

```
                 axis=1  →  resume cada LINHA (somem as colunas)
               ┌─────────────┐
               │  8   6   9  │ → 7.67
   notas  =    │  5   7   6  │ → 6.00
               │ 10   9   8  │ → 9.00
               └─────────────┘
                 ↓   ↓   ↓
  axis=0  →    7.67 7.33 7.67      resume cada COLUNA (somem as linhas)
```

In [ ]:
notas = np.array([[8, 6, 9],
                  [5, 7, 6],
                  [10, 9, 8]])            # 3 alunos (linhas) × 3 provas (colunas)

print("média geral:        ", notas.mean().round(2))
print("média por prova:    ", notas.mean(axis=0).round(2), "→ formato", notas.mean(axis=0).shape)
print("média por aluno:    ", notas.mean(axis=1).round(2), "→ formato", notas.mean(axis=1).shape)
print("melhor prova de cada aluno (posição):", notas.argmax(axis=1))

Uma forma de não se confundir: confira o formato do resultado. A matriz é `(3, 3)` = (alunos, provas). Com `axis=0` a dimensão dos alunos desaparece e sobra um valor por prova.

O argumento `keepdims=True` mantém a dimensão resumida com tamanho 1. Parece inútil, e é exatamente o que faz o *broadcasting* da próxima seção funcionar.

In [ ]:
print(notas.mean(axis=1).shape, "sem keepdims")
print(notas.mean(axis=1, keepdims=True).shape, "com keepdims")

### ✏️ Exercício 3 (★): vendas

`vendas` tem uma linha por loja e uma coluna por mês. Calcule:

1. `total_por_loja`: vetor com o total de cada loja;
2. `media_por_mes`: vetor com a média de cada mês;
3. `melhor_mes_de_cada_loja`: vetor com a posição do mês de maior venda de cada loja;
4. `loja_campea`: a posição da loja com o maior total.

In [ ]:
# ✏️ Exercício 3
vendas = np.array([[120,  90, 150, 110],
                   [ 80, 130,  95, 140],
                   [200, 180, 170, 160]])
# SEU CÓDIGO AQUI
total_por_loja = ...
media_por_mes = ...
melhor_mes_de_cada_loja = ...
loja_campea = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
assert total_por_loja.shape == (3,) and total_por_loja.tolist() == [470, 445, 710], "um total por loja: qual dimensão deve desaparecer?"
assert media_por_mes.shape == (4,) and np.isclose(media_por_mes[0], 400 / 3)
assert melhor_mes_de_cada_loja.tolist() == [2, 3, 0]
assert int(loja_campea) == 2
print("Exercício 3 correto.")

## 5. *Broadcasting*

*Broadcasting* é a regra que permite operar *arrays* de **formatos diferentes**. Você já a usou: em `precos * 0.9`, o número `0.9` foi "esticado" para acompanhar o vetor.

### A regra

Alinhe os formatos **pela direita**. Em cada posição, as dimensões são compatíveis se forem **iguais** ou se uma delas for **1** (ou não existir). A dimensão de tamanho 1 é esticada.

| A | B | Resultado | Por quê |
|---|---|---|---|
| `(3, 4)` | `(4,)` | `(3, 4)` | 4 com 4; a dimensão que falta em B conta como 1 |
| `(3, 4)` | `(3, 1)` | `(3, 4)` | 4 com 1; 3 com 3 |
| `(3, 1)` | `(1, 4)` | `(3, 4)` | 1 com 4; 3 com 1: as duas esticam |
| `(3, 4)` | `(3,)` | **erro** | 4 com 3: nem iguais, nem 1 |

In [ ]:
tabela = np.arange(12).reshape(3, 4)
por_coluna = np.array([10, 20, 30, 40])            # formato (4,): um valor por coluna
por_linha = np.array([[100], [200], [300]])        # formato (3, 1): um valor por linha

print(tabela + por_coluna, "\n")
print(tabela + por_linha)

### 🔮 Preveja antes de rodar

Queremos subtrair de cada linha de `tabela` a média daquela linha. A célula abaixo tenta `tabela - tabela.mean(axis=1)`. Vai funcionar?

In [ ]:
try:
    tabela - tabela.mean(axis=1)
except ValueError as erro:
    print("ValueError:", erro)

A média por linha tem formato `(3,)`. Alinhando pela direita com `(3, 4)`, o NumPy compara 4 com 3 e desiste. É preciso dizer que esses três valores são **uma coluna**, com formato `(3, 1)`. Há três formas equivalentes:

In [ ]:
medias = tabela.mean(axis=1)

a = tabela - tabela.mean(axis=1, keepdims=True)    # a mais legível
b = tabela - medias[:, None]                       # None cria uma dimensão de tamanho 1
c = tabela - medias.reshape(-1, 1)

print(a)
print("as três são iguais?", np.array_equal(a, b) and np.array_equal(b, c))

> ⚠️ **O perigo maior é quando o erro não acontece.** Se a matriz fosse quadrada, `(3, 3)`, a subtração de um vetor `(3,)` funcionaria sem reclamar e subtrairia **por coluna**, e não por linha. O resultado teria o formato certo e os números errados. Confira sempre o formato do que está sendo combinado.

In [ ]:
quadrada = np.array([[1.0, 2.0, 3.0],
                     [10.0, 20.0, 30.0],
                     [100.0, 200.0, 300.0]])

errado = quadrada - quadrada.mean(axis=1)                      # roda, e está errado
certo = quadrada - quadrada.mean(axis=1, keepdims=True)

print("média de cada linha depois de 'centralizar':")
print("  sem keepdims:", errado.mean(axis=1))
print("  com keepdims:", certo.mean(axis=1))

### Aplicação: padronizar colunas

Padronizar é subtrair a média e dividir pelo desvio padrão **de cada coluna**. Usaremos o conjunto *iris* (150 flores, 4 medidas cada), que acompanha o scikit-learn.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target
print("X:", X.shape, "· y:", y.shape, "· classes:", list(iris.target_names))
print("média por coluna:", X.mean(axis=0))

### ✏️ Exercício 4 (★★): padronização

Crie `X_pad`, com as colunas de `X` padronizadas: cada coluna com média 0 e desvio padrão 1. Não use laços.

In [ ]:
# ✏️ Exercício 4
# SEU CÓDIGO AQUI
X_pad = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert X_pad.shape == X.shape
assert np.allclose(X_pad.mean(axis=0), 0) and np.allclose(X_pad.std(axis=0), 1), "média 0 e desvio 1 em cada COLUNA"
print("Exercício 4 correto. Médias:", X_pad.mean(axis=0).round(6), "· desvios:", X_pad.std(axis=0))

**Pergunta escrita.** No exercício 4 não foi preciso `keepdims`. Por quê? Em que situação a mesma expressão, com `axis=1`, exigiria `keepdims=True`?

✏️ **Sua resposta:**

_Escreva aqui._

## 6. Trocar laços por operações vetorizadas

O roteiro para vetorizar é sempre o mesmo: escreva a conta para **um** elemento, veja em que formato os dados precisam estar para o *broadcasting* fazer o resto e confira o formato do resultado.

**Exemplo:** a distância euclidiana de cada flor a uma flor de referência.

In [ ]:
referencia = X[0]                                   # formato (4,)

# Com laço
dist_laco = np.array([np.sqrt(((linha - referencia) ** 2).sum()) for linha in X])

# Vetorizado: (150, 4) - (4,) → (150, 4); soma em axis=1 → (150,)
dist_vetor = np.sqrt(((X - referencia) ** 2).sum(axis=1))

print("iguais?", np.allclose(dist_laco, dist_vetor), "· formato:", dist_vetor.shape)
print("as 3 flores mais próximas da referência:", np.argsort(dist_vetor)[:3])

Um passo além: a distância de **cada** ponto de um conjunto a **cada** ponto de outro. Colocamos um conjunto "em pé" e o outro "deitado", e o *broadcasting* monta todas as combinações.

```
   A[:, None, :]   formato (n, 1, d)
 − B[None, :, :]   formato (1, m, d)
 ────────────────────────────────────
                   formato (n, m, d)   → soma em axis=2 → (n, m)
```

In [ ]:
A_pts, B_pts = X[:5], X[100:103]                    # 5 pontos e 3 pontos

diferencas = A_pts[:, None, :] - B_pts[None, :, :]
distancias = np.sqrt((diferencas ** 2).sum(axis=2))

print("formato das diferenças:", diferencas.shape)
print("matriz de distâncias, 5 × 3:\n", distancias)

### ✏️ Exercício 5 (★★): *softmax*

A função *softmax* transforma uma linha de números quaisquer em probabilidades que somam 1. Ela aparece na saída de toda rede neural de classificação:

$$\text{softmax}(z)_j = \frac{e^{z_j}}{\sum_k e^{z_k}}$$

Escreva `softmax(Z)`, que aplica a função a **cada linha** de uma matriz `Z`, sem laços. Para evitar números gigantescos, subtraia de cada linha o seu valor máximo antes da exponencial (isso não altera o resultado).

In [ ]:
# ✏️ Exercício 5
def softmax(Z):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
_Z = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0], [1000.0, 1001.0, 1002.0]])
_P = softmax(_Z)
assert _P.shape == _Z.shape and np.allclose(_P.sum(axis=1), 1), "cada LINHA deve somar 1"
assert np.allclose(_P[1], 1 / 3) and np.allclose(_P[0], _P[2]), "linhas que diferem por uma constante têm o mesmo softmax"
assert not np.isnan(_P).any(), "apareceu nan: faltou subtrair o máximo de cada linha"
print("Exercício 5 correto:\n", _P)

## 7. Álgebra linear em três minutos

O operador `@` faz o produto de matrizes. Com ele, uma regressão linear cabe em duas linhas: procurar os pesos `w` que deixam `X @ w` o mais próximo possível de `y`.

In [ ]:
rng = np.random.default_rng(0)

x = rng.uniform(0, 10, size=50)
ruido = rng.normal(0, 1, size=50)
alvo = 3 * x + 2 + ruido                             # a "verdade": inclinação 3, intercepto 2

X_reg = np.column_stack([x, np.ones_like(x)])        # coluna de 1s para o intercepto → (50, 2)
pesos, *_ = np.linalg.lstsq(X_reg, alvo, rcond=None) # mínimos quadrados

previsto = X_reg @ pesos                             # (50, 2) @ (2,) → (50,)
print(f"inclinação estimada: {pesos[0]:.2f} · intercepto estimado: {pesos[1]:.2f}")
print(f"erro médio absoluto: {np.abs(alvo - previsto).mean():.2f}")

No produto `A @ B`, o número de colunas de A precisa ser igual ao número de linhas de B: `(n, d) @ (d, k) → (n, k)`. Quando um modelo reclama de formatos incompatíveis, quase sempre é essa regra.

## 8. Sorteios reprodutíveis

O NumPy tem o seu gerador de números aleatórios. Crie um com uma **semente** e use-o para todos os sorteios.

In [ ]:
rng = np.random.default_rng(42)

print("uniforme entre 0 e 1:", rng.random(3))
print("normal, média 0 e desvio 1:", rng.normal(size=3))
print("inteiros de 0 a 9:", rng.integers(0, 10, size=5))
print("permutação de 0 a 5:", rng.permutation(6))
print("amostra sem reposição:", rng.choice(100, size=4, replace=False))

Para embaralhar `X` e `y` **mantendo o pareamento** entre eles, sorteie uma permutação das posições e use-a para indexar os dois.

### ✏️ Exercício 6 (★★): dividir em treino e teste

Escreva `dividir_treino_teste(X, y, proporcao_teste=0.2, semente=None)`, que devolve `X_treino, X_teste, y_treino, y_teste`:

- sorteie uma permutação das posições com `np.random.default_rng(semente)`;
- as primeiras `round(n * proporcao_teste)` posições da permutação formam o teste; as demais, o treino.

In [ ]:
# ✏️ Exercício 6
def dividir_treino_teste(X, y, proporcao_teste=0.2, semente=None):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
_Xa, _Xb, _ya, _yb = dividir_treino_teste(X, y, 0.2, semente=1)
assert _Xa.shape == (120, 4) and _Xb.shape == (30, 4) and _ya.shape == (120,) and _yb.shape == (30,)
_juntos = np.vstack([_Xa, _Xb])
assert np.allclose(np.sort(_juntos, axis=0), np.sort(X, axis=0)), "todas as linhas devem aparecer exatamente uma vez"
_rotulos = {tuple(linha): int(c) for linha, c in zip(X, y)}
assert all(_rotulos[tuple(linha)] == int(c) for linha, c in zip(_Xb, _yb)), "X e y perderam o pareamento"
assert np.array_equal(dividir_treino_teste(X, y, 0.2, semente=1)[1], _Xb), "a mesma semente deve repetir a divisão"
assert len(np.unique(_yb)) > 1, "o teste tem uma única classe: os dados não foram embaralhados"
print("Exercício 6 correto. Classes no teste:", np.bincount(_yb))

## 9. Projeto: k-vizinhos mais próximos em NumPy

O classificador **k-NN** é o mais simples que existe: para classificar um ponto novo, encontre os `k` pontos de treino mais próximos e escolha a classe mais frequente entre eles. Não há treino; a previsão é pura conta de distâncias, perfeita para praticar vetorização.

Vamos preparar os dados e resolver o caso `k = 1` juntos.

In [ ]:
posicoes = np.random.default_rng(42).permutation(len(X))          # 30% para teste: 45 das 150 flores
X_teste, y_teste = X[posicoes[:45]], y[posicoes[:45]]
X_treino, y_treino = X[posicoes[45:]], y[posicoes[45:]]

print("treino:", X_treino.shape, "· teste:", X_teste.shape)

In [ ]:
# Distância de cada ponto de teste a cada ponto de treino: (45, 1, 4) - (1, 105, 4) → (45, 105)
D = np.sqrt(((X_teste[:, None, :] - X_treino[None, :, :]) ** 2).sum(axis=2))

mais_proximo = D.argmin(axis=1)                 # para cada ponto de teste, a posição do vizinho mais próximo
previsto_1nn = y_treino[mais_proximo]           # a classe desse vizinho

print("matriz de distâncias:", D.shape)
print(f"acurácia com k = 1: {(previsto_1nn == y_teste).mean():.3f}")

Para `k > 1` faltam dois passos:

- **os k mais próximos:** `np.argsort(D, axis=1)[:, :k]` devolve, para cada linha, as posições das k menores distâncias;
- **a votação:** com as classes dos vizinhos em uma matriz `(n_teste, k)`, conte os votos de cada classe. Um jeito sem laço usa a codificação *one-hot*: `np.eye(n_classes)[rotulos]` transforma cada rótulo em um vetor com 1 na posição da classe; somando ao longo dos vizinhos, obtém-se a contagem de votos.

In [ ]:
exemplo = np.array([[0, 2, 2],
                    [1, 1, 0]])                 # classes dos 3 vizinhos de 2 pontos
votos = np.eye(3, dtype=int)[exemplo].sum(axis=1)
print("formato do one-hot:", np.eye(3, dtype=int)[exemplo].shape)
print("votos por classe:\n", votos)
print("classe vencedora:", votos.argmax(axis=1))

### ✏️ Exercício 7 (★★★): k-NN completo

Escreva `prever_knn(X_treino, y_treino, X_novos, k=5)`, que devolve a classe prevista para cada linha de `X_novos`, sem laços.

In [ ]:
# ✏️ Exercício 7
def prever_knn(X_treino, y_treino, X_novos, k=5):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 7 (não altere)
from sklearn.neighbors import KNeighborsClassifier

_meu = prever_knn(X_treino, y_treino, X_teste, k=5)
_ref = KNeighborsClassifier(n_neighbors=5).fit(X_treino, y_treino).predict(X_teste)
assert _meu.shape == y_teste.shape, "devolva uma classe por linha de X_novos"
assert (_meu == _ref).mean() >= 0.95, "as previsões diferem muito das do scikit-learn"
assert np.array_equal(prever_knn(X_treino, y_treino, X_teste, k=1), previsto_1nn), "com k=1 o resultado deve coincidir com o da célula guiada"
print(f"Exercício 7 correto. Acurácia com k = 5: {(_meu == y_teste).mean():.3f} · concordância com o scikit-learn: {(_meu == _ref).mean():.0%}")

## 10. Armadilhas numéricas

In [ ]:
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3, "→ use np.isclose:", np.isclose(0.1 + 0.2, 0.3))

com_nan = np.array([1.0, np.nan, 3.0])
print("média com nan:", com_nan.mean(), "· ignorando os nan:", np.nanmean(com_nan))
print("nan == nan ?", np.nan == np.nan, "→ use np.isnan:", np.isnan(com_nan))

pixels = np.array([250, 10], dtype=np.uint8)            # inteiros de 0 a 255, como em imagens
print("uint8: 250 + 10 =", (pixels + np.uint8(10))[0], "(deu a volta!)")

grande = np.float32(16_777_216.0)
print("float32: 16777216 + 1 =", grande + np.float32(1.0), "(a precisão acabou)")

- Decimais são aproximações. Compare com `np.isclose` ou `np.allclose`, nunca com `==`.
- `nan` contamina qualquer conta e não é igual a nada, nem a si mesmo.
- Tipos inteiros pequenos **dão a volta** ao passar do limite, sem aviso. Converta imagens para decimal antes de fazer contas.
- `float32` tem cerca de 7 dígitos significativos; `float64`, cerca de 16.

## 11. Salvar e carregar

In [ ]:
np.save("X_pad_exemplo.npy", X)                         # um array
np.savez("iris_exemplo.npz", X=X, y=y)                  # vários, em um arquivo

carregado = np.load("iris_exemplo.npz")
print(carregado["X"].shape, carregado["y"].shape, "· idêntico ao original:", np.array_equal(carregado["X"], X))

## 12. Para levar

### Erros comuns

| Erro | Como evitar |
|---|---|
| Confundir `(n,)` com `(n, 1)` | imprimir `.shape`; usar `keepdims=True` ou `[:, None]` |
| *Broadcasting* que roda e dá o número errado | conferir o formato dos operandos, sobretudo com matrizes quadradas |
| Alterar o original ao mexer em uma fatia | `.copy()` quando precisar de independência |
| `and` / `or` entre *arrays* | `&`, `|`, `~`, com parênteses |
| Comparar decimais com `==` | `np.isclose` |
| Agregação com `nan` | `np.nanmean`, `np.nansum` ou tratar os ausentes antes |
| Laço elemento a elemento | escrever a conta para o *array* inteiro |

### Resumo

- Um *array* é um bloco de números do mesmo tipo; `shape` e `dtype` dizem quase tudo sobre ele.
- Fatias são **visões**; máscaras e listas de índices produzem **cópias**.
- `axis` indica a dimensão que desaparece na agregação.
- *Broadcasting* alinha formatos pela direita e estica dimensões de tamanho 1.
- Vetorizar deixa o código mais curto, mais rápido e mais parecido com a fórmula.
- Sorteios com gerador próprio e semente tornam o trabalho reprodutível.

### O que vem a seguir

No notebook 02, o pandas acrescenta ao *array* aquilo que falta para trabalhar com dados reais: nomes de colunas, tipos diferentes por coluna, valores ausentes e operações de agrupamento. Máscaras, `axis` e vetorização continuam valendo.

### Para casa

A lista de exercícios está em `01_casa.ipynb`.